In [0]:
from pyspark.sql.functions import input_file_name  #importing necessary functions

In [0]:
spark.conf.set(
"fs.azure.account.key.weatherdatadatalake.dfs.core.windows.net",
"replace_with_your_key"
)

##  Using Auto Loader

In [0]:
df = (
    spark.readStream                              ## reading data as a stream. Spark continuously monitor the folder for new files.
         .format("cloudFiles")                    ## this tells spark to use auto loader    
         .option("cloudFiles.format", "csv")      ## the files we expect are CSV files   
         .option("header", "true")                ## CSV contains header 
         .option("cloudFiles.schemaLocation", "abfss://bronze@weatherdatadatalake.dfs.core.windows.net/schema/weather_csv") 
         .load("abfss://bronze@weatherdatadatalake.dfs.core.windows.net/weather_csv/")
)

In [0]:
new_df = df.withColumn("source_file", input_file_name())

## Storing data as Delta table

In [0]:
(
    new_df.writeStream
        .format("delta")                               ## store dat as delta
        .option("checkpointLocation","abfss://bronze@weatherdatadatalake.dfs.core.windows.net/checkpoints/weather_csv")  ## tracking purpose
        .outputMode("append")
        .start("abfss://bronze@weatherdatadatalake.dfs.core.windows.net/delta/weather_csv_to_delta")  ## destination fro delta table
)

In [0]:
df_delta = spark.read.format("delta")\
            .load("abfss://bronze@weatherdatadatalake.dfs.core.windows.net/delta/weather_csv_to_delta")

display(df_delta)

city,temperature,humidity,weather_condition,_rescued_data,source_file
Hyderabad,32.03,28,clear,null,abfss://bronze@weatherdatadatalake.dfs.core.windows.net/weather_csv/hyderabad_weather%20-%20Sheet1.csv
bangalore,29.3,18,clear,null,abfss://bronze@weatherdatadatalake.dfs.core.windows.net/weather_csv/bangalore_weather%20-%20Sheet1.csv
mumbai,29.32,54,clear,null,abfss://bronze@weatherdatadatalake.dfs.core.windows.net/weather_csv/mumbai_weather%20-%20Sheet1.csv
delhi,35.61,7,clouds,null,abfss://bronze@weatherdatadatalake.dfs.core.windows.net/weather_csv/delhi_weather%20-%20Sheet1.csv
